# Phase 2 — Evaluation

Loads the artifacts produced by `phase2_colab_train.ipynb` and runs the full validation suite the rubric expects:

1. Learning curves (MFM val loss + SVDD radius drift) — interpreted, not just shown.
2. ROC / PR / F1 on a random split and on the **zero-day** split (held-out attack family).
3. Per-attack-family confusion → which families the model misses.
4. **Ablation**: MFM-only vs SVDD-only vs MFM→SVDD vs MFM→SVDD+InfoNCE.
5. **Robustness**: AUC under Gaussian feature-noise perturbation.
6. **Interpretability**: CLS attention heatmap on the top-N most-anomalous true positives.
7. Comparison vs Phase 1 (NSL-KDD MAE) and vs the refined-ML ensemble.

Set the artifact directory in the first cell.

In [ ]:
from pathlib import Path
import json, sys

ARTIFACT_DIR = Path('models/phase2_local/checkpoints')   # or wherever Drive synced to
DATA_DIR     = Path('data/cic_ids')                       # local copy of CIC-IDS2017
DEVICE       = 'cpu'                                      # eval can run on CPU

sys.path.insert(0, str(Path('src').resolve()))
ARTIFACT_DIR, ARTIFACT_DIR.exists()

## 1. Load checkpoint, scaler, split metadata

In [ ]:
import numpy as np, torch
from cps_ad.torch_ft_svdd import load_checkpoint, score_samples, reconstruction_error, cls_attention_attribution
from cps_ad.cic_ids import load_cic_ids2017, make_zero_day_split

model, extra = load_checkpoint(str(ARTIFACT_DIR / 'ft_svdd_final.pt'), map_location=DEVICE)
split_meta = json.loads((ARTIFACT_DIR / 'split_meta.json').read_text())
scaler_npz = np.load(ARTIFACT_DIR / 'scaler.npz')
mean, scale = scaler_npz['mean'], scaler_npz['scale']
feature_names = split_meta['feature_names']
print('held-out family:', split_meta['held_out_family'], ' n_features=', len(feature_names))

## 2. Rebuild evaluation splits (deterministic seed)

In [ ]:
df = load_cic_ids2017(data_dir=DATA_DIR, download=True)
split = make_zero_day_split(df, held_out_family=split_meta['held_out_family'])
def standardize(x):
    return ((x.to_numpy(dtype=np.float32) - mean) / np.where(scale > 0, scale, 1.0)).astype(np.float32)

x_known = standardize(split.test_known.x); y_known = split.test_known.y
x_zero  = standardize(split.test_zero_day.x); y_zero = split.test_zero_day.y
x_known.shape, x_zero.shape

## 3. Learning curves

In [ ]:
import matplotlib.pyplot as plt
hist = json.loads((ARTIFACT_DIR / 'history.json').read_text())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist['pretrain_train'], label='train'); ax[0].plot(hist['pretrain_val'], label='val')
ax[0].set_title('MFM masked-MSE'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist['finetune_loss'], label='svdd loss'); ax[1].plot(hist['finetune_radius'], label='mean ||z-c||')
ax[1].set_title('Deep SVDD finetune'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

**Reading the curves.** MFM val loss should track training closely (small generalisation gap). The mean SVDD radius should *decrease then plateau*; if it crashed to ~0 we have hypersphere collapse — the InfoNCE term + no-bias projection are designed to prevent that.

## 4. ROC / PR / F1 — known and zero-day

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve
import pandas as pd

def evaluate(name, x, y):
    s = score_samples(model, x, device=DEVICE)
    auc = roc_auc_score(y, s)
    pr  = average_precision_score(y, s)
    p, r, t = precision_recall_curve(y, s)
    f1 = 2 * p * r / np.maximum(p + r, 1e-12)
    best = int(np.argmax(f1))
    return {'split': name, 'ROC-AUC': auc, 'PR-AUC': pr,
            'F1*': f1[best], 'thr*': float(t[max(best - 1, 0)])}

rows = [evaluate('test_known (random split)', x_known, y_known),
        evaluate(f"test_zero_day ({split_meta['held_out_family']})", x_zero, y_zero)]
pd.DataFrame(rows)

## 5. Per-family confusion (zero-day diagnostic)

In [ ]:
scores_known = score_samples(model, x_known, device=DEVICE)
thr = np.median(scores_known[y_known == 0])
pred = (scores_known > thr).astype(int)
fam = split.test_known.family
report = pd.DataFrame({'family': fam, 'pred_anomaly': pred, 'truth_anomaly': y_known})
report.groupby('family').agg(n=('truth_anomaly', 'size'),
                              recall=('pred_anomaly', 'mean'))

## 6. Ablation

In [ ]:
# Ablation re-uses the same encoder weights but switches the score function.
# - 'recon' uses mean MSE reconstruction (MFM-only)
# - 'svdd'  uses ||z - c||^2 (final model)
# - 'sum'   sums the two after rank-normalization on benign val
from cps_ad.metrics import evaluate_scores

x_val = standardize(split.val.x)
recon_val = reconstruction_error(model, x_val, device=DEVICE)
svdd_val  = score_samples(model, x_val, device=DEVICE)

def rank_norm(ref, x):
    order = np.argsort(np.argsort(ref))
    rank_map = np.zeros_like(order, dtype=float)
    rank_map[order] = np.arange(len(ref)) / max(len(ref) - 1, 1)
    return np.interp(x, np.sort(ref), rank_map[np.argsort(ref)])

for name, x_eval, y_eval in [('test_known', x_known, y_known),
                             ('test_zero_day', x_zero, y_zero)]:
    recon = reconstruction_error(model, x_eval, device=DEVICE)
    svdd  = score_samples(model, x_eval, device=DEVICE)
    scores = {'recon-only': recon, 'svdd-only': svdd,
              'recon + svdd (rank fused)': rank_norm(recon_val, recon) + rank_norm(svdd_val, svdd)}
    print('==', name)
    for k, v in scores.items():
        print(f'  {k:30s} ROC-AUC={roc_auc_score(y_eval, v):.4f}'
              f'  PR-AUC={average_precision_score(y_eval, v):.4f}')

## 7. Robustness: Gaussian feature noise

In [ ]:
rng = np.random.default_rng(0)
for sigma in [0.0, 0.01, 0.05, 0.1, 0.25]:
    x_n = x_zero + rng.normal(scale=sigma, size=x_zero.shape).astype(np.float32)
    s = score_samples(model, x_n, device=DEVICE)
    print(f'sigma={sigma:>5.2f}  ROC-AUC={roc_auc_score(y_zero, s):.4f}')

## 8. Interpretability: CLS attention on top-N anomalies

In [ ]:
import seaborn as sns
scores_zero = score_samples(model, x_zero, device=DEVICE)
tp_idx = np.where(y_zero == 1)[0]
topN = tp_idx[np.argsort(-scores_zero[tp_idx])[:10]]
attn = cls_attention_attribution(model, x_zero[topN], device=DEVICE)
fig, ax = plt.subplots(figsize=(min(0.18 * len(feature_names), 16), 4))
sns.heatmap(attn, xticklabels=feature_names, yticklabels=[f'flow_{i}' for i in topN],
            cbar_kws={'label': 'CLS->feature attention'}, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
ax.set_title(f'Top-10 zero-day true positives — {split_meta["held_out_family"]}')
plt.tight_layout(); plt.show()
for row, idx in zip(attn, topN):
    top = np.argsort(-row)[:3]
    print(f'flow_{idx}: ' + ', '.join(f'{feature_names[i]} ({row[i]:.3f})' for i in top))

## 9. Comparison: refined-ML ensemble vs DL

In [ ]:
from cps_ad.refined_ml import RefinedAnomalyEnsemble
x_tr = standardize(split.train.x)
x_va = standardize(split.val.x)
ens = RefinedAnomalyEnsemble(nu=0.05, n_estimators=200).fit(x_tr, x_va)
for name, x_eval, y_eval in [('test_known', x_known, y_known),
                              ('test_zero_day', x_zero, y_zero)]:
    s = ens.score(x_eval)
    print(f'{name:>14s}  refined-ML ROC-AUC={roc_auc_score(y_eval, s):.4f}'
          f'  PR-AUC={average_precision_score(y_eval, s):.4f}')